# 01 - Data overview

Describes the sample used by every later notebook. No models are fitted; run time is a few seconds.

* **Figure 1** - NEPSE against the 27 retained foreign markets: rebased levels and daily returns
* **Figure 3** - lead-lag correlation of each foreign market's return with NEPSE's (lags -5 to +5)
* `coverage_screen.csv` - the coverage screen that retains 27 of the 33 foreign markets (Section 2)
* `leadlag_correlations.csv`, `leadlag_ranking.csv` - the numbers behind Figure 3 and Section 4.1

In [ ]:
# Make the project package importable, from this folder or the project root.
# In Colab: mount Drive and set ROOT = Path("/content/drive/MyDrive/NEPSE_replication") instead.
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "nepse").is_dir())
sys.path.insert(0, str(ROOT / "src"))
print("project root:", ROOT)

In [ ]:
from nepse import config, data, plots

config.ensure_dirs()
plots.apply_style()

panel, nepse_close, turnover = data.load_all()
print(f"panel   : {panel.shape[0]} rows x {panel.shape[1]} markets, "
      f"{panel.index.min().date()} to {panel.index.max().date()}")
print(f"NEPSE   : {nepse_close.notna().sum()} observations, "
      f"{nepse_close.index.min().date()} to {nepse_close.index.max().date()}")
print(f"retained: {len(config.SELECTED)} foreign markets + NEPSE")

## Coverage screen

Coverage is measured on *genuine* observations (a level identical to the previous
day counts as no observation), which exposes series that quietly stopped updating.
Six foreign markets fail the screen and are excluded from `config.SELECTED`.
The `Nepal` column of the panel is not used as a predictor.

In [ ]:
coverage = data.coverage_table(panel, nepse_close)
coverage.to_csv(config.TABLE_DIR / "coverage_screen.csv", index=False)
print(coverage.to_string(index=False))

## Figure 1 - NEPSE and the foreign markets

In [ ]:
_ = plots.market_overview(panel, nepse_close)          # fig1_market_overview

## Figure 3 - lead-lag structure

Correlation between each foreign market's daily log return and NEPSE's, at lags
-5 to +5. A positive lag means the foreign market moves first. Levels are not
used: correlating non-stationary index levels would be spurious.

In [ ]:
lag_corr = data.lead_lag_correlations(panel)
_ = plots.leadlag_heatmap(lag_corr)                     # fig3_leadlag_heatmap
lag_corr.round(4).to_csv(config.TABLE_DIR / "leadlag_correlations.csv")

absolute = lag_corr.abs()
print(f"largest |correlation|: {absolute.max().max():.3f}; "
      f"share of cells below 0.06: {(absolute < 0.06).to_numpy().mean():.0%}")
print(f"markets whose largest |correlation| is at a negative lag: "
      f"{int((absolute.idxmax(axis=1).astype(int) < 0).sum())} of {len(absolute)}")

## Ranking by leading correlation (Section 4.1)

Sum of |correlation| over lags 1-5, i.e. the positive-lag columns of Figure 3.

In [ ]:
ranking = data.lead_lag_ranking(lag_corr)
ranking.round(4).to_csv(config.TABLE_DIR / "leadlag_ranking.csv")
print(ranking.round(4).to_string())